In [1]:
defe

NameError: name 'defe' is not defined

In [1]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from geoalchemy2 import Geometry
from shapely.geometry import LineString

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [2]:
PATH_SHPS = RUTA_UNIDAD_ONE_DRIVE + r"\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA\SHP_RECORRIDOS"
ALIAS = ['M-01', 'M-02', 'M-03', 'M-04']

In [3]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def obtener_archivos_db():
    query = text(f"SELECT maquina || '|' || nombre_archivo FROM datos_cosecha.registro_de_archivos")
    try:
        engine = obtener_engine()
        with engine.connect() as conn:
            result = conn.execute(query)
            # Retornamos un set para búsquedas O(1) en Python
            return [row[0] for row in result]
    except Exception as e:
        print(f"Error al consultar la tabla de registro: {e}")
        return []

def obtener_archivos_locales(path_local):
    try:
        # Listamos archivos que terminen en .shp y sean archivos (no carpetas)
        archivos_shp = [
            archivo for archivo in os.listdir(path_local) 
            if archivo.lower().endswith('.shp') and os.path.isfile(os.path.join(path_local, archivo))
        ]
        return archivos_shp
    except FileNotFoundError:
        print(f"Error: La ruta '{path_local}' no existe.")
        return []
    except Exception as e:
        print(f"Error al leer la carpeta: {e}")
        return []

def cargar_archivo_local(ruta_shp):
    gdf = gpd.read_file(ruta_shp)
    gdf['IsoTime'] = pd.to_datetime(gdf['IsoTime'])
    #gdf.columns = [c.lower() for c in gdf.columns]
    gdf.columns = gdf.columns.str.lower()
    gdf = gdf.rename(columns={'geometry': 'geom'})
    gdf = gdf.set_geometry('geom')
    gdf = gdf.to_crs(epsg=32720)
    columnas_db = ['distance',
                   'swathwidth',
                   'vryldrcane',
                   'sectionid',
                   'crop',
                   'trash',
                   'time',
                   'heading',
                   'variety',
                   'elevation',
                   'isotime',
                   'machine',
                   'fuel',
                   'vehiclspeed',
                   'producthash', 
                   'geom']
    gdf = gdf[columnas_db].copy()
    return gdf

def obtener_id_lote(conn, nombre_archivo):
    """Busca el ID de la tabla maestra usando el nombre del archivo."""
    try:
        query = text("""
            SELECT id FROM datos_cosecha.fechas 
            WHERE nombre_archivo = :nom_file
        """)
        res = conn.execute(query, {'nom_file': nombre_archivo}).fetchone()
        return res[0] if res else None
    except (IndexError, ValueError):
        return None

def cargar_puntos_shps(ruta_carpeta):
    """Función principal que coordina la lectura y carga a Postgres."""
    engine = obtener_engine()
    pendientes = obtener_lotes_pendientes() # Tu función que filtra puntos_cargados IS FALSE
    for nombre_archivo in pendientes:
        ruta_shp = os.path.join(ruta_carpeta, f"{nombre_archivo}.shp")
        if not os.path.exists(ruta_shp):
            print(f"⚠️ Archivo no encontrado: {nombre_archivo}")
            continue
        try:
            with engine.begin() as conn:
                # 1. Identificar el lote
                lote_id = obtener_id_lote(conn, nombre_archivo)
                if lote_id is None:
                    print(f"❌ No existe registro maestro para: {nombre_archivo}")
                    continue
                # 2. Preparar los datos
                df_final = preparar_geodataframe_utm(ruta_shp, lote_id)
                # 3. Insertar detalles
                df_final.to_postgis(
                    'recorridos_cosecha', 
                    conn, 
                    schema='datos_cosecha', 
                    if_exists='append'
                )
                # 4. Actualizar estado
                conn.execute(text("""
                    UPDATE datos_cosecha.fechas 
                    SET puntos_cargados = TRUE 
                    WHERE id = :id
                """), {'id': lote_id})
                print(f"✅ {nombre_archivo}: {len(df_final)} puntos cargados.")
        except Exception as e:
            print(f"❌ Error procesando {nombre_archivo}: {str(e)}")

def registrar_log_archivo(nombre_archivo, cantidad_filas, alias):
    query = text("""
        INSERT INTO datos_cosecha.registro_de_archivos 
        (nombre_archivo, fecha_registro, cantidad_registros, maquina)
        VALUES (:nombre, CURRENT_DATE, :cantidad, :alias)
    """)
    try:
        engine = obtener_engine()
        with engine.begin() as conn:
            conn.execute(query, {"nombre": nombre_archivo, "cantidad": cantidad_filas, "alias": alias})
        return True
    except Exception as e:
        print(f"Error al registrar el log del archivo: {e}")
        return False

def insertar_datos_cosecha(gdf, nombre_archivo, alias):
    if gdf.empty:
        print("El GeoDataFrame está vacío.")
        return
    # 1. Nombre de la tabla temporal
    temp_table = "temp_recorrido_cosechadora"
    try:
        engine = obtener_engine()
        with engine.begin() as conn:
            # 2. Cargar el GDF a una tabla temporal (sin las restricciones de la original)
            # Usamos if_exists='replace' para asegurar que la temporal esté limpia
            gdf.to_postgis(temp_table, conn, if_exists='replace', index=False)
            # 3. Insertar desde la temporal a la real usando ON CONFLICT DO NOTHING
            # Esto valida contra el UNIQUE CONSTRAINT (machine, isotime)
            query = text(f"""
                INSERT INTO datos_cosecha.recorrido_cosechadora 
                (distance, swathwidth, vryldrcane, sectionid, crop, trash, time, 
                 heading, variety, elevation, isotime, machine, fuel, 
                 vehiclspeed, producthash, distancia_real, diferencia_distancia, geom, alias, lote, propiedad)
                SELECT 
                    distance, swathwidth, vryldrcane, sectionid, crop, trash, time, 
                    heading, variety, elevation, isotime, machine, fuel, 
                    vehiclspeed, producthash, distancia_real, diferencia_distancia, geom, alias, lote, propiedad
                FROM {temp_table}
                ON CONFLICT (alias, isotime) DO NOTHING;
            """)
            result = conn.execute(query)
            # 4. Eliminar la tabla temporal
            conn.execute(text(f"DROP TABLE IF EXISTS {temp_table}"))
        registrar_log_archivo(nombre_archivo, len(gdf), alias)
        print(f"Inserción completada. Filas procesadas: {len(gdf)}")
        return True
    except Exception as e:
        print(f"Error durante la inserción: {e}")
        return False

In [4]:
archivos_procesados = obtener_archivos_db()
print('Lista de archivos procesados en base de datos:', len(archivos_procesados))

lista_archivos_locales = []
for maquina in ALIAS:
    ruta = PATH_SHPS + '\\' + maquina
    archivos_locales = obtener_archivos_locales(ruta)
    archivos_locales = [ maquina + '|' + file_name for file_name in archivos_locales]
    lista_archivos_locales = lista_archivos_locales + archivos_locales
    print(f'Lista de archivos en carpera local {maquina}:', len(archivos_locales))

#print('Lista de archivos en carpera local M-01:', len(lista_archivos_locales))

archivos_pendientes = list(set(lista_archivos_locales) - set(archivos_procesados))
print('Lista de archivos en pendientes:', len(archivos_pendientes))

Lista de archivos procesados en base de datos: 80
Lista de archivos en carpera local M-01: 20
Lista de archivos en carpera local M-02: 20
Lista de archivos en carpera local M-03: 20
Lista de archivos en carpera local M-04: 20
Lista de archivos en pendientes: 0


In [5]:
archivos_pendientes

[]

In [6]:
datos_procesados = []

for nombre_archivo in archivos_pendientes:
    # Tomamos lo que está a la derecha del "|"
    parte_derecha = nombre_archivo.split('|')[1]
    
    # Separamos por "_"
    columnas = parte_derecha.split('_')
    
    # Guardamos el resultado en nuestra lista
    datos_procesados.append(columnas)

# 3. Creación de la tabla (DataFrame)
# Definimos nombres de columnas basados en la estructura de tus archivos
nombres_columnas = ['Productor', 'Finca', 'Lote', 'Operacion', 'Fecha', 'Codigo_Ext']
df = pd.DataFrame(datos_procesados, columns=nombres_columnas)

# 4. Mostrar el resultado
df

,Productor,Finca,Lote,Operacion,Fecha,Codigo_Ext


In [7]:
for archivo in archivos_pendientes:
    alias = archivo.split('|')[0]
    file_name = archivo.split('|')[1]

    datos_archivo = file_name.split('_')
    canero = datos_archivo[0]
    propiedad = datos_archivo[1]
    lote = datos_archivo[2]
    #arma la ruta del archivo
    ruta_shp = os.path.join(PATH_SHPS, alias, file_name)
    
    print(ruta_shp)
    
    recorridos_shp = cargar_archivo_local(ruta_shp)

    recorridos_shp['alias'] = alias
    recorridos_shp['lote'] = lote
    recorridos_shp['propiedad'] = propiedad
    
    # 1. Preparación: Convertir a datetime y ordenar cronológicamente
    recorridos_shp['isotime'] = pd.to_datetime(recorridos_shp['isotime'])
    recorridos_shp = recorridos_shp.sort_values('isotime').reset_index(drop=True)
    # 2. Creamos una columna auxiliar con la geometría del punto ANTERIOR
    recorridos_shp['prev_geom'] = recorridos_shp['geom'].shift(1)
    # 3. Filtramos la primera fila (ya que no tiene un punto anterior para formar una línea)
    df_segmentos = recorridos_shp.dropna(subset=['prev_geom']).copy()
    # 4. Generamos los segmentos (LineString) uniendo el punto anterior con el actual
    # Al hacerlo así, la fila mantiene sus datos originales (los del punto de destino)
    df_segmentos['geom'] = df_segmentos.apply(
        lambda x: LineString([x['prev_geom'], x['geom']]), axis=1
    )
    # 5. Limpieza: Eliminamos la columna auxiliar
    df_segmentos = df_segmentos.drop(columns=['prev_geom'])
    
    df_segmentos['distancia_real'] = df_segmentos.geom.length
    df_segmentos['diferencia_distancia'] = (df_segmentos['distance'] - df_segmentos['distancia_real']).abs()

    estado = insertar_datos_cosecha(df_segmentos, file_name, alias)
    print(estado, 'datos registrados de: ', archivo)